# Part 1 - K-Means and Its Variations

This notebook demonstrates the core K-Means algorithm and a few important variations:
- Lloyd's K-Means from scratch
- scikit-learn KMeans
- K-Means++ initialization
- Bisecting K-Means
- MiniBatch K-Means
- Common K-Means failure cases


### Before this cell
- This is the starting point of the notebook.
- We first need data, libraries, and helper functions before running K-Means.

### What this cell does
- Imports NumPy, Pandas, Matplotlib, SciPy, and scikit-learn clustering utilities.
- Creates `X_blobs`, which contains four compact clusters for normal K-Means experiments.
- Creates `X_moons`, which will later show a case where K-Means struggles.
- Defines `plot_clusters()` for visualization and `sse()` for measuring clustering error.
- Uses `random_state=42` so the same data is generated each time.

### What we achieve after running it
- The datasets and helper functions required by the rest of the notebook are ready.
- The first plot lets us visually see the four natural groups in the blob dataset.

### Remember
`X_blobs` = easy case for K-Means.  
`X_moons` = difficult/non-convex case used later.


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.spatial.distance import cdist
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, BisectingKMeans, MiniBatchKMeans
from sklearn.metrics import adjusted_rand_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

X_blobs, y_blobs = make_blobs(
    n_samples=600,
    centers=4,
    cluster_std=0.9,
    random_state=RANDOM_STATE
)

X_moons, y_moons = make_moons(
    n_samples=400,
    noise=0.07,
    random_state=RANDOM_STATE
)

def plot_clusters(X, labels=None, centers=None, title=""):
    plt.figure(figsize=(5.5, 4))
    if labels is None:
        plt.scatter(X[:, 0], X[:, 1], s=20, alpha=0.8)
    else:
        plt.scatter(X[:, 0], X[:, 1], c=labels, cmap="tab10", s=20, alpha=0.8)
    if centers is not None:
        plt.scatter(
            centers[:, 0], centers[:, 1],
            marker="X", s=220, c="black",
            edgecolor="white"
        )
    plt.title(title)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.show()

def sse(X, labels, centers):
    return float(np.sum((X - centers[labels]) ** 2))

print("Blob dataset:", X_blobs.shape)
plot_clusters(X_blobs, y_blobs, title="Synthetic blobs — true groups")


## 1. Lloyd's K-Means from Scratch

K-Means repeatedly performs two main steps:

1. Assign every point to its nearest centroid.
2. Recompute each centroid as the mean of the points assigned to it.

The process stops when the centroids no longer move significantly.


### What we saw before
- The previous cell only prepared the data and helper functions.
- We could see that `X_blobs` naturally contains four groups.

### How this cell is different
- Now we implement the actual K-Means algorithm ourselves instead of using scikit-learn.

### What this cell does
- `cdist(X, centers, "sqeuclidean")` calculates the distance from every point to every centroid.
- `D.argmin(axis=1)` assigns each point to its nearest centroid.
- `X[labels == k].mean(axis=0)` moves each centroid to the mean of the points assigned to it.
- The assign-and-update steps repeat until the centroids stop moving significantly.

### What we achieve after running it
- We get cluster labels, final centroid locations, number of iterations, and SSE.
- We can visually see the clusters discovered by our own implementation.

### Remember
Basic K-Means loop:

**Assign points → Update centroids → Repeat until stable.**


In [ ]:
def kmeans_scratch(X, init, max_iter=100, tol=1e-6):
    centers = np.array(init, dtype=float).copy()

    for iteration in range(1, max_iter + 1):

        # ASSIGN: distance from every point to every centroid
        D = cdist(X, centers, "sqeuclidean")
        labels = D.argmin(axis=1)

        # UPDATE: centroid becomes mean of its assigned points
        new_centers = centers.copy()
        for k in range(len(centers)):
            if np.any(labels == k):
                new_centers[k] = X[labels == k].mean(axis=0)

        shift = np.max(np.linalg.norm(new_centers - centers, axis=1))
        centers = new_centers

        if shift < tol:
            break

    return labels, centers, iteration

# Deliberately poor initialization: four low points
bad_init = X_blobs[np.argsort(X_blobs[:, 1])[:4]]

labels_scratch, centers_scratch, n_iter = kmeans_scratch(
    X_blobs,
    bad_init
)

print("Iterations:", n_iter)
print("Final SSE:", round(sse(X_blobs, labels_scratch, centers_scratch), 2))

plot_clusters(
    X_blobs,
    labels_scratch,
    centers_scratch,
    "Lloyd's K-Means from Scratch"
)


## 2. scikit-learn KMeans and K-Means++

The scikit-learn implementation performs the same basic Lloyd loop.

K-Means++ improves initialization by spreading the initial centroids across the data instead of choosing all starting points completely at random.


### What we saw before
- In the previous section, we manually implemented Lloyd's K-Means.
- We started from deliberately poor centroid positions and repeatedly assigned and updated clusters.

### How this cell is different
- Here we use scikit-learn's ready-made `KMeans` implementation.
- We also compare different initialization strategies instead of focusing only on the iteration logic.

### What this cell does
- `km_same` uses the same poor starting centroids as our scratch implementation.
- `km_random` chooses initial centroids randomly.
- `km_pp` uses `init="k-means++"` with multiple initial attempts.
- SSE measures how tightly points are grouped around their centroids.
- ARI compares the discovered clusters with the known true groups in this synthetic dataset.

### What we achieve after running it
- We verify that scikit-learn performs the same basic K-Means task.
- We can compare how initialization affects clustering quality.
- We see why K-Means++ is generally a better default initialization method.

### Remember
K-Means can reach different solutions depending on where centroids start.  
**K-Means++ tries to choose better-spread starting centroids.**


In [ ]:
# Same poor initialization as the scratch implementation
km_same = KMeans(
    n_clusters=4,
    init=bad_init,
    n_init=1,
    algorithm="lloyd",
    random_state=RANDOM_STATE
).fit(X_blobs)

# Random initialization
km_random = KMeans(
    n_clusters=4,
    init="random",
    n_init=1,
    random_state=RANDOM_STATE
).fit(X_blobs)

# K-Means++ initialization
km_pp = KMeans(
    n_clusters=4,
    init="k-means++",
    n_init=10,
    random_state=RANDOM_STATE
).fit(X_blobs)

comparison = pd.DataFrame({
    "Method": ["Scratch / bad start", "sklearn / same start", "Random init", "K-Means++"],
    "SSE": [
        sse(X_blobs, labels_scratch, centers_scratch),
        km_same.inertia_,
        km_random.inertia_,
        km_pp.inertia_
    ],
    "ARI vs true groups": [
        adjusted_rand_score(y_blobs, labels_scratch),
        adjusted_rand_score(y_blobs, km_same.labels_),
        adjusted_rand_score(y_blobs, km_random.labels_),
        adjusted_rand_score(y_blobs, km_pp.labels_)
    ]
})

display(comparison.round(3))

plot_clusters(
    X_blobs,
    km_pp.labels_,
    km_pp.cluster_centers_,
    "K-Means with K-Means++ Initialization"
)


## 3. Bisecting K-Means

Bisecting K-Means starts with one cluster and repeatedly splits an existing cluster into two until the requested number of clusters is reached.


### What we saw before
- Standard K-Means tried to create all requested clusters together.
- We also saw that initialization can affect the final result.

### How this cell is different
- Bisecting K-Means uses a hierarchical splitting approach.
- It starts with one cluster and repeatedly divides existing clusters until the requested number is reached.

### What this cell does
- Creates data with clusters of different sizes and spreads.
- Tests two strategies:
  - `biggest_inertia`: split the cluster with the highest clustering error.
  - `largest_cluster`: split the cluster containing the most samples.
- Displays SSE and ARI for both strategies.

### What we achieve after running it
- We can compare two ways of deciding which cluster should be split next.
- We see that Bisecting K-Means follows a different clustering strategy while still using K-Means ideas internally.

### Remember
Standard K-Means: **form K clusters together.**  
Bisecting K-Means: **start with one cluster and repeatedly split.**


In [ ]:
X_mix, y_mix = make_blobs(
    n_samples=[400, 100, 100],
    centers=[[0, 0], [9, 0], [9, 8]],
    cluster_std=[0.6, 2.2, 0.5],
    random_state=RANDOM_STATE
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, strategy in zip(
    axes,
    ["biggest_inertia", "largest_cluster"]
):
    bk = BisectingKMeans(
        n_clusters=3,
        bisecting_strategy=strategy,
        random_state=RANDOM_STATE
    ).fit(X_mix)

    ax.scatter(
        X_mix[:, 0], X_mix[:, 1],
        c=bk.labels_, cmap="tab10",
        s=20, alpha=0.8
    )
    ax.scatter(
        bk.cluster_centers_[:, 0],
        bk.cluster_centers_[:, 1],
        marker="X", s=180, c="black"
    )
    ax.set_title(
        f"{strategy}\n"
        f"SSE={bk.inertia_:.1f}, "
        f"ARI={adjusted_rand_score(y_mix, bk.labels_):.2f}"
    )

plt.tight_layout()
plt.show()


## 4. MiniBatch K-Means

MiniBatch K-Means updates centroids using small subsets of the data instead of using the entire dataset on every iteration. This is useful for larger datasets.


### What we saw before
- Bisecting K-Means changed how clusters are created.
- So far, the examples were small enough that processing the full dataset was inexpensive.

### How this cell is different
- Here the focus changes from clustering strategy to scalability and execution time.
- We compare normal K-Means with `MiniBatchKMeans` on 50,000 samples.

### What this cell does
- Standard `KMeans` uses the full dataset during centroid updates.
- `MiniBatchKMeans` uses smaller batches of 1,024 samples at a time.
- `time.perf_counter()` measures execution time.
- SSE and ARI are also recorded so we can compare quality as well as speed.

### What we achieve after running it
- We can see whether MiniBatch K-Means is faster on a larger dataset.
- We can also check whether that speed comes with any change in clustering quality.

### Remember
MiniBatch K-Means is mainly useful when the dataset becomes large:

**smaller updates → better scalability → usually similar clustering quality.**


In [ ]:
X_big, y_big = make_blobs(
    n_samples=50_000,
    centers=8,
    n_features=10,
    cluster_std=1.5,
    random_state=RANDOM_STATE
)

models = [
    (
        "KMeans",
        KMeans(
            n_clusters=8,
            n_init=1,
            random_state=RANDOM_STATE
        )
    ),
    (
        "MiniBatchKMeans",
        MiniBatchKMeans(
            n_clusters=8,
            batch_size=1024,
            n_init=1,
            random_state=RANDOM_STATE
        )
    )
]

results = []

for name, model in models:
    start = time.perf_counter()
    model.fit(X_big)
    elapsed = time.perf_counter() - start

    results.append({
        "Model": name,
        "Seconds": elapsed,
        "SSE": model.inertia_,
        "ARI vs truth": adjusted_rand_score(y_big, model.labels_)
    })

display(pd.DataFrame(results).round(3))


## 5. Where K-Means Can Fail

K-Means works best when clusters are roughly compact, round, and reasonably similar in size and density.

It can struggle with curved or non-convex cluster shapes.


### What we saw before
- Previous sections showed several successful K-Means approaches:
  - Lloyd's K-Means
  - K-Means++
  - Bisecting K-Means
  - MiniBatch K-Means
- All of those examples mostly used compact blob-like clusters.

### How this cell is different
- Instead of showing another improvement, this cell shows an important limitation of K-Means.
- We deliberately use cluster shapes that violate K-Means' assumptions.

### What this cell does
- Tests K-Means on two-moons data.
- Creates concentric-ring data.
- Compares the true groups with the clusters found by K-Means.
- ARI gives a numerical indication of how different the K-Means result is from the true grouping.

### What we achieve after running it
- We see that K-Means can split curved shapes incorrectly even when the correct number of clusters is provided.
- This demonstrates that choosing a clustering algorithm should depend on the structure of the data.

### Remember
K-Means works best for roughly **compact, round, similarly scaled clusters**.

It can struggle with **curved, non-convex, differently sized, or differently dense clusters**.


In [ ]:
# Two classic non-convex examples
rng = np.random.default_rng(RANDOM_STATE)

angles = rng.uniform(0, 2 * np.pi, 600)
radius = np.r_[
    np.full(300, 1.0),
    np.full(300, 3.5)
] + rng.normal(0, 0.12, 600)

X_rings = np.c_[
    radius * np.cos(angles),
    radius * np.sin(angles)
]
y_rings = np.repeat([0, 1], 300)

cases = [
    ("Two Moons", X_moons, y_moons, 2),
    ("Concentric Rings", X_rings, y_rings, 2)
]

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

for j, (name, X, y, k) in enumerate(cases):
    model = KMeans(
        n_clusters=k,
        n_init=10,
        random_state=RANDOM_STATE
    ).fit(X)

    axes[0, j].scatter(X[:, 0], X[:, 1], c=y, cmap="tab10", s=15)
    axes[0, j].set_title(f"{name}: true groups")

    axes[1, j].scatter(X[:, 0], X[:, 1], c=model.labels_, cmap="tab10", s=15)
    axes[1, j].scatter(
        model.cluster_centers_[:, 0],
        model.cluster_centers_[:, 1],
        marker="X", s=170, c="black"
    )
    axes[1, j].set_title(
        f"K-Means result — ARI "
        f"{adjusted_rand_score(y, model.labels_):.2f}"
    )

plt.tight_layout()
plt.show()


## Conclusion

This notebook demonstrated the core K-Means assign/update loop and three useful variations:

- K-Means++ improves centroid initialization.
- Bisecting K-Means builds clusters through repeated splitting.
- MiniBatch K-Means trades some precision for better scalability.

The final examples also show that K-Means can struggle with non-convex cluster shapes such as moons and rings.
